In [1]:
%load_ext cudf.pandas

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [3]:

# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
import pandas as pd
# from tpch import load_lineitem, load_supplier, load_orders, load_customer, load_nation, q07
DATA_ROOT = "/home/jupyter/tpch-dbgen/data"
STORAGE_OPTS = {}


In [4]:

### cell 0 ###

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)


Index(['L_ORDERKEY', 'L_PARTKEY', 'L_SUPPKEY', 'L_LINENUMBER', 'L_QUANTITY',
       'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_TAX', 'L_RETURNFLAG',
       'L_LINESTATUS', 'L_SHIPDATE', 'L_COMMITDATE', 'L_RECEIPTDATE',
       'L_SHIPINSTRUCT', 'L_SHIPMODE', 'L_COMMENT', 'L_DUMMY'],
      dtype='object')


In [5]:

### cell 1 ###

def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)



In [6]:

### cell 2 ###

def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)



In [7]:

### cell 3 ###


def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
customer = load_customer(DATA_ROOT, **STORAGE_OPTS)


In [8]:

### cell 4 ###

def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [9]:

# 1. Filter, compute year & volume, project only needed columns
li = (
    lineitem
    .loc[
        (lineitem.L_SHIPDATE >= "1995-01-01") &
        (lineitem.L_SHIPDATE <  "1997-01-01")
    ]
    .assign(
        L_YEAR=lambda df: df.L_SHIPDATE.dt.year,
        VOLUME=lambda df: df.L_EXTENDEDPRICE * (1 - df.L_DISCOUNT)
    )[['L_ORDERKEY','L_SUPPKEY','L_YEAR','VOLUME']]
)

# 2. Pre-filter nation table
nations = nation[
    nation.N_NAME.isin(["FRANCE","GERMANY"])
][['N_NATIONKEY','N_NAME']]

# 3. Attach nation to customer & supplier with minimal columns
cust = (
    customer[['C_CUSTKEY','C_NATIONKEY']]
    .merge(nations, left_on='C_NATIONKEY', right_on='N_NATIONKEY', how='inner')
    [['C_CUSTKEY','N_NAME']]
    .rename(columns={'N_NAME':'CUST_NATION'})
)

supp = (
    supplier[['S_SUPPKEY','S_NATIONKEY']]
    .merge(nations, left_on='S_NATIONKEY', right_on='N_NATIONKEY', how='inner')
    [['S_SUPPKEY','N_NAME']]
    .rename(columns={'N_NAME':'SUPP_NATION'})
)

# 4. Join lineitem → orders → customer → supplier, then filter
df = (
    li
    .merge(orders[['O_ORDERKEY','O_CUSTKEY']], left_on='L_ORDERKEY', right_on='O_ORDERKEY', how='inner')
    .merge(cust, left_on='O_CUSTKEY', right_on='C_CUSTKEY', how='inner')
    .merge(supp, left_on='L_SUPPKEY', right_on='S_SUPPKEY', how='inner')
    [['SUPP_NATION','CUST_NATION','L_YEAR','VOLUME']]
)

# 5. Keep only France↔Germany pairs
df = df[
    ((df.SUPP_NATION == 'FRANCE') & (df.CUST_NATION == 'GERMANY')) |
    ((df.SUPP_NATION == 'GERMANY') & (df.CUST_NATION == 'FRANCE'))
]


In [ ]:
# 6. Aggregate on GPU and rename
total = (
    df
    .groupby(['SUPP_NATION','CUST_NATION','L_YEAR'], as_index=False)['VOLUME']
    .sum()
    .rename(columns={'VOLUME':'REVENUE'})
)